# SourceStretching — the M1 induction source term $S=(b\cdot\nabla)u$

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``source_stretching``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

The `sources { induction_stretching { } }` block adds $S=(b\cdot\nabla)u$ to
the right-hand side. The module transports a single scalar component, so with
a uniform $b=B_0$ and a **divergence-free** velocity $u=(x,-y)$ (so
$\nabla u=\mathrm{diag}(1,-1)$), convection and diffusion vanish identically
and

$$\frac{d b}{dt}=B_0\frac{\partial u_0}{\partial x}=B_0,$$

i.e. $b(t)=B_0e^{t}$. A wrong factor, a transposed gradient index, or a
dropped term changes the exponential rate, so this is a sharp absolute check
of the source magnitude and sign.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| — | M1 stretching source $S=(b\cdot\nabla)u$ | $b\sim e^{t}$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Sequential-only: the dataset is run by `trustutils` on one processor (no `partition()`, no `nbProcs`).


In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "source_stretching.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("source_stretching_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
t, b0 = read_son("source_stretching", "SONDE_B0")
b0_0 = b0[0]
expected = b0_0 * np.exp(t)
err = np.max(np.abs(b0 - expected)) / abs(b0_0)
print(f"b0(0)={b0_0:.4f}  b0(t_end)={b0[-1]:.4f}  expected exp(t_end)={expected[-1]:.4f}")
print(f"max |b - B0 exp(t)| / B0 : {err:.3e}")
# the source must reproduce exp(t) to a few percent (first-order in dt
# for an explicit term; dt=2e-3 keeps the error well below 1%)
assert err < 0.01, f"induction source rate wrong: {err:.3e}"
print("OK: M1 source S=(b.grad)u has the correct magnitude and sign.")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(t, b0 / b0_0, label="b/B0 (simulated)")
plt.plot(t, np.exp(t), "k--", label="exp(t) (analytic)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()